Carregando os dados e separando df's de treino e de teste.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dengue_pipeline").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from dengue_pipeline import DengueDataCleaner
from dengue_pipeline.models import GradientBoostingDiseaseClassifier
from dengue_pipeline.paths import MODELS_DIR, MODEL_FIGURES_DIR

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def salvar_modelo(modelo, nome_arquivo):
    caminho = MODELS_DIR / nome_arquivo
    joblib.dump(modelo, caminho)
    print(f"Modelo salvo em: {caminho}")
    return caminho

def plotar_importancias(importancias, titulo, nome_arquivo, top_n=30):
    dados = importancias.sort_values(ascending=False).head(top_n).sort_values()
    fig, ax = plt.subplots(figsize=(11, 8))
    ax.barh(dados.index, dados.values, color="#2563eb", alpha=0.9)
    ax.set_title(titulo, fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel("Importância")
    ax.set_ylabel("")
    ax.grid(axis="x", linestyle="--", alpha=0.22)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    caminho = MODEL_FIGURES_DIR / nome_arquivo
    fig.savefig(caminho, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Gráfico salvo em: {caminho}")

cleaner = DengueDataCleaner()
df = cleaner.transformar_ml()

df_train = df[(df["notification_year"].isin([2017, 2018])) | ((df["notification_year"] == 2019) & (df["notification_month"] <= 5))].copy()
df_test = df[(df["notification_year"] == 2019) & (df["notification_month"] >= 6)].copy()

print("Treino:", df_train.shape)
print("Teste:", df_test.shape)

print("Proporção treino:", (len(df_train) / len(df))*100)
print("Proporção teste:", (len(df_test) / len(df))*100)

df.columns

Definindo o target e eliminando colunas que não são apropriadas para o modelo

In [ ]:
target = "final_classification"

y_train = df_train[target]
y_test = df_test[target]

cols_remove = ['final_classification']

X_train = df_train.drop(columns = cols_remove)
X_test = df_test.drop(columns=cols_remove)

X_train = X_train.select_dtypes(include=["number"])
X_test = X_test[X_train.columns]

X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

In [ ]:
administrative_features = [
    "notification_year",
    "notification_month",
    "symptom_epi_year",
    "notif_municipality",
    "notif_health_region",
    "health_facility",
]

X_train = X_train.drop(columns=administrative_features, errors="ignore")
X_test = X_test.drop(columns=administrative_features, errors="ignore")

Implementação de Regressão Logistica 

In [ ]:
#Regressao Logistica - testando...
from sklearn.linear_model import LogisticRegression

modelo_logistic = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

modelo_logistic.fit(X_train, y_train)

y_pred_logistic = modelo_logistic.predict(X_test)

print(classification_report(y_test, y_pred_logistic))
print(confusion_matrix(y_test, y_pred_logistic))

salvar_modelo(modelo_logistic, "logistic_regression.joblib")
importancias_logistic = pd.Series(
    modelo_logistic.named_steps["classifier"].coef_[0],
    index=X_train.columns,
).abs().sort_values(ascending=False)
plotar_importancias(
    importancias_logistic,
    "Regressão Logística - Top 30 Features",
    "logistic_regression_feature_importance.png",
)

Implementação de XGBoost

In [ ]:
# Teste da classe adaptada usando XGBoost.
# O XGBoost trata valores ausentes nativamente.
modelo_xgb = GradientBoostingDiseaseClassifier(
    model="xgb",
    fast_train=False,
    device="cuda"
)

modelo_xgb.fit(
    X_train,
    y_train,
    n_trials=200,
    tuning_sample_size=200_000,
)
y_pred_xgb = modelo_xgb.predict(X_test)

print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

resultados_limiares_xgb = modelo_xgb.evaluate(
    X_test,
    y_test,
    thresholds=[0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80],
)
display(resultados_limiares_xgb)

salvar_modelo(modelo_xgb, "xgboost.joblib")
importancias_xgb = modelo_xgb.feature_importance(importance_type="gain")
plotar_importancias(
    importancias_xgb,
    "XGBoost - Top 30 Features por Ganho",
    "xgboost_feature_importance.png",
)

In [ ]:
# Teste da classe adaptada usando LightGBM.
# O LightGBM trata valores ausentes nativamente.
modelo_lgb = GradientBoostingDiseaseClassifier(
    model="lgbm",
    fast_train=False,
    device="cuda"
)

modelo_lgb.fit(
    X_train,
    y_train,
    n_trials=200,
    tuning_sample_size=200_000,
)
y_pred_lgb = modelo_lgb.predict(X_test)

print(classification_report(y_test, y_pred_lgb))
print(confusion_matrix(y_test, y_pred_lgb))

resultados_limiares_lgb = modelo_lgb.evaluate(
    X_test,
    y_test,
    thresholds=[0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80],
)
display(resultados_limiares_lgb)

salvar_modelo(modelo_lgb, "lightgbm.joblib")
importancias_lgb = modelo_lgb.feature_importance(importance_type="gain")
plotar_importancias(
    importancias_lgb,
    "LightGBM - Top 30 Features por Ganho",
    "lightgbm_feature_importance.png",
)

Implementação de Arvore de Decisao

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelo_decision_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=10,
        class_weight="balanced",
        random_state=42
    ))
])

modelo_decision_tree.fit(X_train, y_train)

y_pred_decision_tree = modelo_decision_tree.predict(X_test)

print(classification_report(y_test, y_pred_decision_tree))
print(confusion_matrix(y_test, y_pred_decision_tree))

salvar_modelo(modelo_decision_tree, "decision_tree.joblib")
importancias_decision_tree = pd.Series(
    modelo_decision_tree.named_steps["classifier"].feature_importances_,
    index=X_train.columns,
).sort_values(ascending=False)
plotar_importancias(
    importancias_decision_tree,
    "Árvore de Decisão - Top 30 Features",
    "decision_tree_feature_importance.png",
)